# Index

In [1]:
import os
import uuid
import ast

from consts import *

import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

### Chunk the articles

In [3]:
df = pd.read_csv(SUBSET_DATASET_FILENAME)

In [4]:
def _safe_list(value):
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []

documents = []

print("Splitting articles intelligently via structural boundaries...")
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name=EMBEDDING_BASE_MODEL,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=int(CHUNK_SIZE * OVERLAP_RATIO),
    separators=["\n\n", "\n", " ", ""],
)

for _, row in df.iterrows():
    article_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, str(row['url'])))
    article_text_chunks = splitter.split_text(str(row['text']))
    chunk_count = 1
    
    for chunk_text in article_text_chunks:
        documents.append(
            Document(
                page_content=chunk_text,
                metadata={
                    "article_id": article_id,
                    "chunk_id": chunk_count,
                    "title": str(row['title']),
                    "url": str(row['url']),
                    "authors": _safe_list(row["authors"]),
                    "tags": _safe_list(row["tags"]),
                }
            )
        )
        chunk_count += 1

print(f"Created {len(documents)} chunks")

Splitting articles intelligently via structural boundaries...
Created 19 chunks


### Embed the chunks

In [5]:
embeddings_model = OpenAIEmbeddings(
    api_key=os.environ["LLMOD_API_KEY"],
    base_url=LLMOD_BASE_URL,
    model=EMBEDDING_MODEL
)

metadatas = [doc.metadata for doc in documents]
texts = [doc.page_content for doc in documents]
vectors = embeddings_model.embed_documents(texts)

### Create a Pinecone index

In [6]:
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
spec = ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION)
    
if PINECONE_INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=len(vectors[0]),
        metric="cosine",
        spec=spec
    )
pinecone_index = pc.Index(PINECONE_INDEX_NAME)

print(f"{pinecone_index} index status:", end='\n\n')
print(pinecone_index.describe_index_stats(), end='\n\n')

<pinecone.db_data.index.Index object at 0x114903450> index status:

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}



### Upsert embedings to Pinecone

In [7]:
records = []

for doc, vector in zip(documents, vectors):
    records.append({
        "id": f"{doc.metadata['article_id']}:{doc.metadata['chunk_id']}",
        "values": vector,
        "metadata": {
            **doc.metadata,
            "text": doc.page_content,
        },
    })

pinecone_index.upsert(vectors=records, namespace=PINECONE_NAMESPACE)
print(pinecone_index.describe_index_stats(), end='\n\n')

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'Medium-Rag': {'vector_count': 19}},
 'total_vector_count': 19,
 'vector_type': 'dense'}



In [18]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_pinecone import PineconeVectorStore

In [11]:
from langchain_core.prompts import ChatPromptTemplate

retrieval_prompt = ChatPromptTemplate.from_messages([
    ("system", LLM_SYSTEM_PROMPT + "\n\nContext:\n{context}"),
    ("human", "{input}"),
])

In [ ]:
from collections import defaultdict

"""
The spec only cares about ARTICLES, not chunks. 
The example questions are "which article" (find one, list 3, summarize one, recommend one).
So article score = its highest schored chunk.
Filtering multiple chunks retrived from the same article remains different articles instead of different pieces from the same one.
"""

# config — these will live in your single config object later
FETCH_K            = 20   # chunks to pull BEFORE grouping (wide net)
MIN_ARTICLES       = 3    # distinct articles to keep (narrow result)
CHUNKS_PER_ARTICLE = 2    # best chunks to keep within each kept article

def retrieve_distinct(question, n_articles=3, chunks_per_article=2, pool=20):
    seen, selected = [], []
    for _ in range(n_articles):
        flt = {"article_id": {"$nin": seen}} if seen else None
        scored = vectorstore.similarity_search_with_score(question, k=pool, filter=flt)
        if not scored:
            break  # no more relevant articles exist — return what we have
        top_article = scored[0][0].metadata["article_id"]   # best remaining article
        chunks = [(d, s) for d, s in scored if d.metadata["article_id"] == top_article]
        chunks.sort(key=lambda p: p[1], reverse=True)
        selected.extend(chunks[:chunks_per_article])
        seen.append(top_article)
    return selected
    
def retrieve(question, fetch_k=FETCH_K, max_articles=MIN_ARTICLES,
             chunks_per_article=CHUNKS_PER_ARTICLE):
    # 1. Over-fetch chunks WITH scores. This is the wide net.
    scored = vectorstore.similarity_search_with_score(question, k=fetch_k)
    # scored = [(Document, score), ...]; for your Pinecone cosine setup,
    # higher score = more relevant (your earlier output confirmed: 0.50 top, 0.05 bottom)

    # 2. Group the fetched chunks by their source article.
    by_article = defaultdict(list)
    for doc, score in scored:
        by_article[doc.metadata["article_id"]].append((doc, score))

    # 3. Rank ARTICLES by their single best chunk (max-pooling).
    ranked_articles = sorted(
        by_article.items(),
        key=lambda item: max(score for _, score in item[1]),
        reverse=True,   # higher = better; flip only if you ever switch to a distance metric
    )

    # 4. Keep the top N articles; within each, keep its best M chunks.
    selected = []
    for article_id, chunk_list in ranked_articles[:max_articles]:
        chunk_list.sort(key=lambda pair: pair[1], reverse=True)  # best chunks first
        selected.extend(chunk_list[:chunks_per_article])

    return selected   # [(Document, score), ...], article-ranked and grouped

In [19]:
vectorstore = PineconeVectorStore(
    index=pinecone_index,
    embedding=embeddings_model,
    namespace=PINECONE_NAMESPACE,
    text_key="text",
)

retriever=vectorstore.as_retriever(
    search_kwargs={
        "k": TOP_K
    }
)

llm = ChatOpenAI(    
    api_key=os.environ["LLMOD_API_KEY"],
    base_url=LLMOD_BASE_URL,
    model=LLM_MODEL
)

document_prompt = PromptTemplate.from_template(
    "Title: {title}\n"
    "Authors: {authors}\n"
    "URL: {url}\n"
    "Tags: {tags}\n"
    "Passage: {page_content}"
)

combine_docs_chain = create_stuff_documents_chain(
    llm, 
    retrieval_prompt,
    document_prompt=document_prompt
)
retrieval_chain = create_retrieval_chain(retriever, combine_docs_chain)

In [31]:
def answer(user_prompt, top_k=TOP_K):
    scored = vectorstore.similarity_search_with_score(user_prompt, k=top_k)

    context_str = "\n\n".join(
        document_prompt.format(**{**doc.metadata, "page_content": doc.page_content})
        for doc, _ in scored
    )

    system_prompt = LLM_SYSTEM_PROMPT + "\n\nContext:\n" + context_str

    # 4. call the model
    response = llm.invoke([("system", system_prompt), ("human", user_prompt)])

    return {
        "response": response.content,
        "context": [
            {
                "article_id": doc.metadata["article_id"],
                "title": doc.metadata["title"],
                "chunk": doc.page_content,
                "score": float(score),
            }
            for doc, score in scored
        ],
        "Augmented_prompt": {"System": system_prompt, "User": user_prompt},
    }

In [32]:
answer(USER_QUERIES[0])

{'response': 'Title: How to Turn Your Popular Blog Series Into a Bestselling Book\nAuthor: Frank Mckinley\n\nWhy this fits: Frank McKinley reframes marketing as dialogue and service to readers — e.g., “As you blog, invite your readers to talk to you. Use that feedback…” and later: “Finally, it’s not all about you … You can set yourself apart by focusing on service.” He explicitly addresses writer discomfort with self-promotion: “With all that going for you, you’ll have no guilt. Marketing won’t feel so slimy. You know your book is good, and you can stand behind it with honest pride. Make it about them and they’ll make your dream come true.” These passages show the article positions marketing as a conversation aimed at writers uneasy about self-promotion.',
 'context': [{'article_id': '3b9df0d8-033a-52c9-985b-8efa04658d60',
   'title': 'How to Turn Your Popular Blog Series Into a Bestselling Book',
   'chunk': 'Hope\n\nAn escape\n\nChances are, they want a mix of these things. As you bl

In [22]:
result = retrieval_chain.invoke(
    #{"input": USER_QUERIES[1]}
    {"input": "List exactly 3 articles. Return only the titles."}
)
print(result["answer"])

How to Turn Your Popular Blog Series Into a Bestselling Book
Occam’s dice
Mind Your Nose


In [23]:
result = retrieval_chain.invoke(
    {"input": USER_QUERIES[2]}
)
print(result["answer"])

I don’t know based on the provided Medium articles data.

I searched the supplied article passages and metadata (including "Your Brain On Coronavirus" by Simon Spichak, "Occam’s dice" by Kelly Clancy, "Mental Note Vol. 24" and others). None of those pieces argue that past pandemics like the bubonic plague spurred innovation and recovery. For example, "Your Brain On Coronavirus" discusses neurological and psychiatric effects of COVID-19 (strokes, long-haulers, worsening mental health) but does not make a historical-innovation argument; "Occam’s dice" discusses simplicity and biological contingency (evolution rolls a die) but not pandemics driving recovery; and "Mental Note Vol. 24" only notes that “for some of us, this pandemic‑imposed isolation is a boon rather than a trial,” which is not the same as a claim about historical pandemics stimulating societal innovation. Because no article in the provided dataset makes the specific argument you ask about, I can’t summarize such an article 

In [24]:
result = retrieval_chain.invoke(
    {"input": USER_QUERIES[3]}
)
print(result["answer"])

Recommend: "Mind Your Nose" (author: Ann-Sophie Barwich) — because it gives a concrete, beginner-friendly practice routine you can model for habit formation.

Why (based on the article):
- The piece describes a clear, repeatable schedule: "This entire exercise was undertaken each day for 20 minutes during the six weeks."  
- It breaks the practice into simple, doable tasks (classification, identification, detection), so a beginner knows exactly what to do.  
- The practice was monitored: "Responses were monitored and evaluated on speed and accuracy," which shows the value of tracking progress.  
- The article reports measurable payoff: "Such intense olfactory training led to a general improvement in olfactory performance" and that improvements "transferred to other olfactory abilities" — evidence that a short, regular routine produced lasting change.

Practical, article-based steps you can use to build any habit (taking the article as a model):
1. Make it short and specific — e.g., a 2

In [33]:
def stats():
    return {
        "chunk_size": CHUNK_SIZE,
        "overlap_ratio": OVERLAP_RATIO,
        "top_k": TOP_K
    }

In [36]:
stats()

{'chunk_size': 512, 'overlap_ratio': 0.1, 'top_k': 20}